# 04 · Pre-registered confirmation study (Colab T4)

Tests the hypothesis in `PREREGISTRATION.md` on assets/periods never used before.
**Do not change code or configs between steps.** Run every scoring step first, the report last.

Colab Secrets (with *Notebook access* on for this notebook): `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`, `HF_TOKEN`.
After a disconnect: rerun **Setup**, then the step you were on (everything resumes).

## Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'

# Unpack code from Drive. Set UPDATE = True once after uploading a new kronos-agent.zip.
import os, shutil, zipfile, sys
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/kronos_agent/confirm.py'):
    assert os.path.exists(ZIP), f'upload kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)

%pip install -q ccxt yfinance bitsandbytes accelerate
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)
for k in ('ALPACA_API_KEY', 'ALPACA_SECRET_KEY', 'HF_TOKEN'):
    os.environ[k] = userdata.get(k)

EQ = ' '.join(f'configs/confirm/{t}.yaml' for t in ['qqq', 'iwm', 'aapl', 'msft', 'nvda', 'amzn', 'tsla', 'jpm'])
CR = 'configs/confirm/btc.yaml configs/confirm/eth.yaml'
!nvidia-smi -L
print(ROOT, CODE)

## Step 1 · Prices + news for all 10 assets, one example prompt each (CPU, ~10 min)

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --dry-run --root $ROOT --config $EQ $CR 2>&1 | grep -E '^\[|Error|error'

## Step 2 · PRIMARY: real headlines (GPU, ~1 h)

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --variant real --root $ROOT --config $EQ $CR

## Step 3 · Placebo: shuffled headlines (GPU, ~1 h)

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --variant shuffled --root $ROOT --config $EQ $CR

## Step 4 · Anonymized prompts (GPU, ~1 h)

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --variant anon --root $ROOT --config $EQ $CR

## Step 5 · Report (CPU, ~1 min) — run once, after Steps 2–4

In [ ]:
!cd $CODE && python scripts/run_confirm.py --root $ROOT --equities $EQ --crypto $CR